# Reproducibility notebook — *Do Text-to-Music Models Take Direction?*
Regenerates every statistic, table, and figure in the paper from the released CSVs.

Expected repo layout (run from the repository root):
```
data/scored_full_v2.csv        # per-clip scores, raw + loudness-normalized lenses
data/s6_*.csv, data/s7_*.csv   # banked analysis outputs (cross-check targets)
data/features_verbal.csv   # optional: per-clip acoustic descriptors (deep repro)
figures/                       # written by this notebook
```
All bootstrap confidence intervals: 2,000 clip-level resamples, percentile method, seed 0 per call — outputs are digit-identical across runs and match the paper.

In [1]:

import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

B, SEED = 2000, 0
DATA, FIGS = "data", "figures"
os.makedirs(FIGS, exist_ok=True)

df = pd.read_csv(f"{DATA}/scored_full_v2.csv")
assert len(df) == 1500 and df.file.is_unique
SV = df[df.phrasing == "verbal"]
MODELS = sorted(df.model.unique())
LEVELS = [1, 3, 5, 7, 9]

def opt(path):
    return pd.read_csv(path) if os.path.exists(path) else None
cis_bank  = opt(f"{DATA}/s6_bootstrap_cis.csv")
decomp    = opt(f"{DATA}/s6_feature_decomp.csv")
raw_lvl   = opt(f"{DATA}/s6_raw_ridge_level_means.csv")
feats     = opt(f"{DATA}/features_verbal.csv")
s7_cis    = opt(f"{DATA}/s7_bootstrap_cis.csv")

def ci(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    return np.percentile(a, 2.5), np.percentile(a, 97.5)

def boot_rho(lv, sc):
    rng = np.random.default_rng(SEED)
    lv, sc = np.asarray(lv), np.asarray(sc); n = len(lv)
    idx = rng.integers(0, n, (B, n))
    return np.array([spearmanr(lv[i], sc[i])[0] for i in idx])

def boot_stat(arrs, fn):
    rng = np.random.default_rng(SEED)
    idxs = [rng.integers(0, len(a), (B, len(a))) for a in arrs]
    return np.array([fn([a[ix[b]] for a, ix in zip(arrs, idxs)]) for b in range(B)])

print(f"loaded {len(df)} clips | models: {MODELS}")


loaded 1500 clips | models: ['ace', 'mgm', 'mgs', 'sa3s', 'sao']


In [2]:

# ---- Table 1: monotonicity, phrasing contrast, dynamic range (median + mean) ----
rows = []
for m in MODELS:
    r = {}
    for ph in ["numeric", "verbal"]:
        g = df[(df.model == m) & (df.phrasing == ph)]
        rho = spearmanr(g.level, g.p0a)[0]
        lo, hi = ci(boot_rho(g.level.values, g.p0a.values))
        r[f"rho_{ph}"] = f"{rho:+.3f} [{lo:+.3f},{hi:+.3f}]"
        if ph == "verbal":
            a1 = g[g.level == 1].p0a.values; a9 = g[g.level == 9].p0a.values
            md = np.median(a9) - np.median(a1)
            mlo, mhi = ci(boot_stat([a1, a9], lambda xs: np.median(xs[1]) - np.median(xs[0])))
            mn = a9.mean() - a1.mean()
            nlo, nhi = ci(boot_stat([a1, a9], lambda xs: xs[1].mean() - xs[0].mean()))
            r["median_delta"] = f"{md:+.2f} [{mlo:+.2f},{mhi:+.2f}]"
            r["mean_delta"] = f"{mn:+.2f} [{nlo:+.2f},{nhi:+.2f}]"
    gn = df[(df.model == m) & (df.phrasing == "numeric")]
    gv = df[(df.model == m) & (df.phrasing == "verbal")]
    drho = spearmanr(gv.level, gv.p0a)[0] - spearmanr(gn.level, gn.p0a)[0]
    dlo, dhi = ci(boot_rho(gv.level.values, gv.p0a.values) - boot_rho(gn.level.values, gn.p0a.values))
    r["delta_rho_vn"] = f"{drho:+.3f} [{dlo:+.3f},{dhi:+.3f}]"
    rows.append(dict(model=m, **r))
t1 = pd.DataFrame(rows).set_index("model")
print("TABLE 1 (paper §4.1)"); print(t1.to_string())

if cis_bank is not None:   # cross-check against the banked canonical values
    chk = cis_bank[(cis_bank.analysis == "rho_level_p0a") & (cis_bank.scope == "verbal")]
    for _, row in chk.iterrows():
        g = df[(df.model == row.model) & (df.phrasing == "verbal")]
        assert abs(spearmanr(g.level, g.p0a)[0] - row.point) < 1e-3
    print("\ncross-check vs s6_bootstrap_cis.csv: PASS")


TABLE 1 (paper §4.1)
                  rho_numeric              rho_verbal         median_delta           mean_delta            delta_rho_vn
model                                                                                                                  
ace    +0.070 [-0.089,+0.228]  +0.328 [+0.167,+0.476]  +0.10 [-0.05,+0.70]  +0.18 [-0.12,+0.47]  +0.258 [+0.063,+0.451]
mgm    +0.038 [-0.134,+0.202]  +0.538 [+0.414,+0.644]  +1.30 [+0.50,+2.10]  +1.28 [+0.91,+1.63]  +0.500 [+0.333,+0.668]
mgs    +0.041 [-0.127,+0.197]  +0.388 [+0.247,+0.512]  +2.05 [+0.80,+2.40]  +0.93 [+0.40,+1.45]  +0.347 [+0.199,+0.497]
sa3s   +0.035 [-0.139,+0.198]  +0.384 [+0.223,+0.525]  +1.60 [+0.55,+2.20]  +0.83 [+0.31,+1.33]  +0.350 [+0.222,+0.482]
sao    +0.007 [-0.165,+0.164]  +0.504 [+0.367,+0.619]  +1.30 [+0.55,+2.00]  +1.13 [+0.78,+1.50]  +0.497 [+0.370,+0.627]

cross-check vs s6_bootstrap_cis.csv: PASS


In [3]:

# ---- Table 2: mechanism (content share CIs recomputed; knob shares from bank) ----
print("TABLE 2 (paper §4.2)")
for m in MODELS:
    g = SV[SV.model == m]
    A1 = g[g.level == 1][["p0a", "p0a_n"]].values
    A9 = g[g.level == 9][["p0a", "p0a_n"]].values
    raw_d = A9[:, 0].mean() - A1[:, 0].mean()
    rd_ci = ci(boot_stat([A1, A9], lambda xs: xs[1][:, 0].mean() - xs[0][:, 0].mean()))
    if rd_ci[0] <= 0 <= rd_ci[1]:
        print(f"  {m:5s} content share: undefined (raw-delta CI covers 0)")
        continue
    share = (A9[:, 1].mean() - A1[:, 1].mean()) / raw_d
    slo, shi = ci(boot_stat([A1, A9], lambda xs:
        (xs[1][:, 1].mean() - xs[0][:, 1].mean()) / (xs[1][:, 0].mean() - xs[0][:, 0].mean())))
    print(f"  {m:5s} content share {share:+.2f} [{slo:+.2f},{shi:+.2f}]")
if decomp is not None:
    print("\nfeature-cluster shares of the pre-calibration delta (banked):")
    print(decomp[["model", "d_raw_total", "loudness_share", "noisiness_share",
                  "tonality_share"]].round(3).to_string(index=False))


TABLE 2 (paper §4.2)
  ace   content share: undefined (raw-delta CI covers 0)
  mgm   content share +0.76 [+0.63,+0.91]
  mgs   content share +0.76 [+0.47,+0.99]
  sa3s  content share +0.82 [+0.64,+1.15]
  sao   content share +0.59 [+0.50,+0.69]

feature-cluster shares of the pre-calibration delta (banked):
model  d_raw_total  loudness_share  noisiness_share  tonality_share
  ace        0.208           0.094            0.259           0.647
  mgm        1.857          -0.019            0.623           0.397
  mgs        0.965           0.207            0.266           0.527
 sa3s        0.728           0.183            0.391           0.426
  sao        1.389           0.384            0.380           0.236


/tmp/ipykernel_644/475636360.py:14: RuntimeWarning: divide by zero encountered in scalar divide
  (xs[1][:, 1].mean() - xs[0][:, 1].mean()) / (xs[1][:, 0].mean() - xs[0][:, 0].mean())))


In [4]:

# ---- Figure 1: compliance curves with 95% bootstrap bands ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0), sharey=True)
colors = dict(zip(MODELS, plt.rcParams["axes.prop_cycle"].by_key()["color"]))
for ax, ph in zip(axes, ["numeric", "verbal"]):
    for m in MODELS:
        mu, lo, hi = [], [], []
        for L in LEVELS:
            a = df[(df.model == m) & (df.phrasing == ph) & (df.level == L)].p0a.values
            mu.append(a.mean())
            l, h = ci(boot_stat([a], lambda xs: xs[0].mean()))
            lo.append(l); hi.append(h)
        ax.plot(LEVELS, mu, "-o", ms=4, color=colors[m], label=m)
        ax.fill_between(LEVELS, lo, hi, color=colors[m], alpha=0.15, lw=0)
    ax.set_title(f"{ph} prompts"); ax.set_xlabel("requested energy level")
    ax.set_xticks(LEVELS); ax.grid(alpha=0.25)
axes[0].set_ylabel("Arousal Index (0–11)"); axes[1].legend(fontsize=8)
fig.suptitle("Compliance curves — mean with 95% clip-level bootstrap bands", y=1.02)
fig.tight_layout(); fig.savefig(f"{FIGS}/compliance_curves_ci.png", dpi=200, bbox_inches="tight")
print(f"wrote {FIGS}/compliance_curves_ci.png")


wrote figures/compliance_curves_ci.png


In [5]:

# ---- Per-genre verbal curves (5 x 5 grid) ----
genres = sorted(df.genre_slug.unique())
fig, axes = plt.subplots(len(MODELS), len(genres), figsize=(13, 10),
                         sharex=True, sharey=True)
for i, m in enumerate(MODELS):
    for j, g in enumerate(genres):
        sub = SV[(SV.model == m) & (SV.genre_slug == g)]
        mu = [sub[sub.level == L].p0a.mean() for L in LEVELS]
        axes[i, j].plot(LEVELS, mu, "-o", ms=3)
        if i == 0: axes[i, j].set_title(g, fontsize=9)
        if j == 0: axes[i, j].set_ylabel(m, fontsize=9)
        axes[i, j].grid(alpha=0.25)
fig.suptitle("Verbal compliance by model × genre (mean Arousal Index)")
fig.tight_layout(); fig.savefig(f"{FIGS}/per_genre_curves.png", dpi=160, bbox_inches="tight")
print(f"wrote {FIGS}/per_genre_curves.png")


wrote figures/per_genre_curves.png


In [6]:

# ---- Droop panel: three witnesses at the top of the scale (paper §4.4) ----
print("verbal L9 − L7, 95% CIs")
for m in MODELS:
    g7 = SV[(SV.model == m) & (SV.level == 7)]
    g9 = SV[(SV.model == m) & (SV.level == 9)]
    out = []
    for col, lab in [("p0a", "calibrated"), ("clap", "own-prompt CLAP")]:
        pt = g9[col].mean() - g7[col].mean()
        lo, hi = ci(boot_stat([g7[col].values, g9[col].values],
                              lambda xs: xs[1].mean() - xs[0].mean()))
        out.append(f"{lab} {pt:+.3f} [{lo:+.3f},{hi:+.3f}]")
    print(f"  {m:5s} " + " | ".join(out))
if feats is not None:
    print("\nraw (pre-calibration) witness: recomputable from data/features_verbal.csv")
elif raw_lvl is not None:
    r = raw_lvl.set_index("model")
    print("\nraw (pre-calibration) level means (banked; CIs in s6_bootstrap_cis.csv):")
    print(r.round(2).to_string())


verbal L9 − L7, 95% CIs
  ace   calibrated -0.553 [-0.823,-0.250] | own-prompt CLAP -0.081 [-0.132,-0.027]
  mgm   calibrated +0.433 [+0.150,+0.747] | own-prompt CLAP +0.093 [+0.015,+0.163]
  mgs   calibrated -0.420 [-0.857,+0.027] | own-prompt CLAP -0.046 [-0.120,+0.030]
  sa3s  calibrated -0.303 [-0.690,+0.057] | own-prompt CLAP -0.001 [-0.086,+0.085]
  sao   calibrated +0.233 [+0.030,+0.440] | own-prompt CLAP +0.067 [-0.000,+0.136]

raw (pre-calibration) witness: recomputable from data/features_verbal.csv


In [7]:

# ---- Plausibility lens: bands, edge cells, step-gain contrasts (paper §4.3) ----
BANDS = {"folk": (0.5, 5.5), "edm": (4, 8), "hiphop": (2, 7),
         "orchestral": (3, 7), "rock": (5, 8)}   # rock analyst-set (H1 v2)
STEPS = [(1, 3), (3, 5), (5, 7), (7, 9)]
inb = lambda g, L: BANDS[g][0] <= L <= BANDS[g][1]

def contrast(sub, genres):
    cells = {(g, L): gg.p0a.values
             for (g, L), gg in sub.groupby(["genre_slug", "level"]) if g in genres}
    keys = sorted(cells); arrs = [cells[k] for k in keys]
    pos = {k: i for i, k in enumerate(keys)}
    def gap(xs, which):
        gains = [(xs[pos[(g, b)]].mean() - xs[pos[(g, a)]].mean()) / (b - a)
                 for g in genres for a, b in STEPS
                 if (inb(g, a) and inb(g, b)) == which and (g, a) in pos]
        return float(np.mean(gains))
    pt = gap(arrs, True) - gap(arrs, False)
    lo, hi = ci(boot_stat(arrs, lambda xs: gap(xs, True) - gap(xs, False)))
    return pt, lo, hi

for label, genres in [("all genres", list(BANDS)), ("rock excluded", [g for g in BANDS if g != "rock"])]:
    pt, lo, hi = contrast(SV, genres)
    print(f"pooled in-band minus out-of-band gain, {label}: {pt:+.3f} [{lo:+.3f},{hi:+.3f}]")

edge = SV.groupby(["model", "genre_slug", "level"]).p0a.mean().round(2).unstack()
print("\nper-cell verbal means (see paper App. A; '*' marking in data/s6_edge_cells.csv):")
print(edge.to_string())


pooled in-band minus out-of-band gain, all genres: +0.190 [+0.096,+0.279]


pooled in-band minus out-of-band gain, rock excluded: +0.217 [+0.111,+0.321]

per-cell verbal means (see paper App. A; '*' marking in data/s6_edge_cells.csv):
level                1     3     5     7     9
model genre_slug                              
ace   edm         4.65  4.28  4.48  5.20  4.38
      folk        3.77  3.77  4.02  4.62  3.87
      hiphop      4.57  3.87  4.37  5.10  4.35
      orchestral  3.70  3.90  4.10  4.40  4.05
      rock        4.13  4.20  4.67  5.18  5.08
mgm   edm         5.87  6.38  6.18  6.32  6.50
      folk        4.43  4.03  5.82  4.90  6.25
      hiphop      5.38  4.65  6.32  6.33  6.50
      orchestral  3.77  4.67  4.97  5.98  6.47
      rock        6.37  6.38  6.48  6.50  6.48
mgs   edm         5.40  5.73  6.47  6.42  6.38
      folk        4.15  4.08  4.73  5.72  5.00
      hiphop      4.98  5.53  6.18  6.28  6.33
      orchestral  3.90  3.97  4.02  5.67  4.72
      rock        5.37  6.17  6.33  6.48  6.03
sa3s  edm         4.87  5.80  6.47  6.45  

In [8]:

# ---- Skew exhibit: clip-level scores at L1 vs L9 (why median > mean for mgs) ----
fig, axes = plt.subplots(1, len(MODELS), figsize=(13, 3.2), sharey=True)
rng = np.random.default_rng(0)
for ax, m in zip(axes, MODELS):
    for L, x0 in [(1, 0), (9, 1)]:
        a = SV[(SV.model == m) & (SV.level == L)].p0a.values
        ax.scatter(x0 + rng.uniform(-0.12, 0.12, len(a)), a, s=10, alpha=0.6)
        ax.hlines([np.median(a), a.mean()], x0 - 0.2, x0 + 0.2,
                  colors=["k", "r"], lw=[1.6, 1.0])
    ax.set_xticks([0, 1]); ax.set_xticklabels(["L1", "L9"]); ax.set_title(m)
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Arousal Index")
fig.suptitle("Clip-level scores, verbal L1 vs L9 (black = median, red = mean)", y=1.04)
fig.tight_layout(); fig.savefig(f"{FIGS}/skew_L1L9.png", dpi=160, bbox_inches="tight")
print(f"wrote {FIGS}/skew_L1L9.png")


wrote figures/skew_L1L9.png


In [9]:

# ---- Leaderboard (banked) + cross-check against recomputation ----
lead = opt(f"{DATA}/s7_leaderboard.csv")
if lead is not None:
    print("LEADERBOARD (data/s7_leaderboard.csv):")
    print(lead.to_string(index=False))
    for _, r in lead.iterrows():   # verify banked rho against this notebook's own compute
        g = SV[SV.model == r.model]
        assert abs(spearmanr(g.level, g.p0a)[0] - r.rho_verbal) < 1e-3
    print("\ncross-check (rho_verbal vs recomputed): PASS")
else:
    print("s7_leaderboard.csv not present")


LEADERBOARD (data/s7_leaderboard.csv):
model  house_level  rho_verbal  rho_lo  rho_hi  med_delta  md_lo  md_hi content_share  cs_lo  cs_hi
  ace         4.35       0.328   0.167   0.476       0.10  -0.05    0.7     undefined  -8.07   9.47
  mgm         5.83       0.538   0.414   0.644       1.30   0.50    2.1          0.76   0.63   0.91
  mgs         5.81       0.388   0.247   0.512       2.05   0.80    2.4          0.76   0.47   0.99
 sa3s         5.80       0.384   0.223   0.525       1.60   0.55    2.2          0.82   0.64   1.15
  sao         6.04       0.504   0.367   0.619       1.30   0.55    2.0          0.59   0.50   0.69

cross-check (rho_verbal vs recomputed): PASS


In [10]:

# ---- Raw (pre-calibration) scale replicate of Table 1 (banked CIs) ----
if s7_cis is not None and (s7_cis.analysis == "rho_level_raw").any():
    sub = s7_cis[s7_cis.analysis.isin(["rho_level_raw", "delta_raw_mean", "delta_raw_median"])]
    piv = sub.pivot(index="model", columns="analysis", values=["point", "lo", "hi"]).round(3)
    print("RAW-SCALE REPLICATE (paper §4.4; ceiling-free):")
    for m in piv.index:
        f = lambda a: f'{piv.loc[m, ("point", a)]:+.2f} [{piv.loc[m, ("lo", a)]:+.2f},{piv.loc[m, ("hi", a)]:+.2f}]'
        print(f"  {m:5s} rho {f('rho_level_raw')}   d_mean {f('delta_raw_mean')}   d_median {f('delta_raw_median')}")
else:
    print("raw-scale CI rows not present in s7_bootstrap_cis.csv")


RAW-SCALE REPLICATE (paper §4.4; ceiling-free):
  ace   rho +0.34 [+0.18,+0.49]   d_mean +0.21 [-0.03,+0.43]   d_median +0.23 [-0.03,+0.62]
  mgm   rho +0.59 [+0.47,+0.68]   d_mean +1.86 [+1.47,+2.24]   d_median +1.86 [+1.42,+2.40]
  mgs   rho +0.41 [+0.27,+0.53]   d_mean +0.96 [+0.49,+1.42]   d_median +1.20 [+0.47,+1.66]
  sa3s  rho +0.41 [+0.25,+0.54]   d_mean +0.73 [+0.31,+1.13]   d_median +0.83 [+0.32,+1.33]
  sao   rho +0.54 [+0.40,+0.65]   d_mean +1.39 [+1.03,+1.75]   d_median +1.50 [+0.96,+1.82]


In [11]:

# ---- Fixed-anchor CLAP axis (activates when the axis CSVs are present) ----
axis_lm = opt(f"{DATA}/s7_axis_level_means.csv")
axis_pc = opt(f"{DATA}/s7_clap_axis.csv")
if axis_lm is not None:
    alm = axis_lm.set_index(["model", "phrasing", "level"])["axis"].unstack("level")
    print("axis means by level (verbal):")
    print(alm.xs("verbal", level="phrasing").round(4).to_string())
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    for m in MODELS:
        ax.plot(LEVELS, alm.loc[(m, "verbal")].values, "-o", ms=4, label=m)
    ax.set_xlabel("requested energy level"); ax.set_ylabel("CLAP axis (sim_hi − sim_lo)")
    ax.set_xticks(LEVELS); ax.grid(alpha=0.25); ax.legend(fontsize=8)
    ax.set_title("Fixed-anchor CLAP axis, verbal (genre-matched anchors)")
    fig.tight_layout(); fig.savefig(f"{FIGS}/clap_axis_verbal.png", dpi=160, bbox_inches="tight")
    print(f"wrote {FIGS}/clap_axis_verbal.png")
    if s7_cis is not None and (s7_cis.analysis == "rho_level_axis").any():
        print("\nrho(level, axis) and axis droop (banked, 95% CIs):")
        for a in ["rho_level_axis", "droop_L9-L7_axis"]:
            for _, r in s7_cis[s7_cis.analysis == a].iterrows():
                print(f"  {a:18s} {r.model:5s} {r.scope:8s} {r.point:+.4f} [{r.lo:+.4f},{r.hi:+.4f}]")
else:
    print("CLAP-axis files pending (s7_axis_level_means.csv / s7_clap_axis.csv):")
    print("drop them into data/ from the canonical commit and re-run this notebook.")


axis means by level (verbal):
level       1       3       5       7       9
model                                        
ace   -0.1255 -0.1524 -0.1013 -0.0723 -0.0832
mgm   -0.0016  0.0060  0.0914  0.1060  0.1543
mgs   -0.0266 -0.0104  0.0703  0.1030  0.1131
sa3s  -0.0193 -0.0267  0.0788  0.0947  0.1013
sao   -0.0367  0.0016  0.0416  0.0445  0.0963
wrote figures/clap_axis_verbal.png

rho(level, axis) and axis droop (banked, 95% CIs):
  rho_level_axis     ace   numeric  -0.0112 [-0.1759,+0.1573]
  rho_level_axis     ace   verbal   +0.2743 [+0.1145,+0.4263]
  rho_level_axis     mgm   numeric  -0.0507 [-0.2073,+0.1076]
  rho_level_axis     mgm   verbal   +0.5961 [+0.4768,+0.6902]
  rho_level_axis     mgs   numeric  -0.0797 [-0.2385,+0.0764]
  rho_level_axis     mgs   verbal   +0.5460 [+0.4287,+0.6493]
  rho_level_axis     sa3s  numeric  +0.0113 [-0.1647,+0.1809]
  rho_level_axis     sa3s  verbal   +0.4502 [+0.2986,+0.5903]
  rho_level_axis     sao   numeric  +0.0208 [-0.1544,+0.1785]
  r

## Notes
- Arousal Index and LUFS values are bit-reproducible across scoring runs; own-prompt CLAP shows run-to-run cell-mean drift ≤ 0.033 (released values are from the canonical run; the fixed-anchor CLAP axis in `data/s7_*.csv` is the deterministic replacement).
- Deep reproduction (re-scoring audio from scratch) uses the released Kaggle scoring notebook; this notebook reproduces all *statistics* from released CSVs.